### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in a subsequent processing. Langchain supports multiple schema types and methods for enforcing structured ouput.

### Pydantic

pydantic models provide the richest features set with field validation, description and nested structures

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="Director of the movie")
    rating: float = Field(description="Rating out of 10")

model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [6]:
class FootballPlayer(BaseModel):
    Name: str = Field(description="Name of the player")
    Age : int = Field(description="Age of player")
    club : str = Field(description="Current club of the player")
    Nation : str = Field(description="Player's Nationality")

Player_details = model.with_structured_output(FootballPlayer)
Player_details.invoke("Provide me the player details of Cristiano Ronaldo on the year 2018")

FootballPlayer(Name='Cristiano Ronaldo', Age=33, club='Real Madrid, Juventus', Nation='Portugal')

Enforces types strictly (e.g., title must be a string or it errors) — this is Pydantic's field validation.
include_raw=True returns both the raw LLM message and the parsed structured object.
Nested structures supported — e.g., a MovieDetails model containing cast: list[Actor] where Actor is itself a BaseModel.

### Nested Structure

In [ ]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year : int
    cast: list[Actor]
    genres: list[str]
    budget:  float | None = Field(None, description="Budget in million USD")

ActorMovieDetails = model.with_structured_output(MovieDetails)
ActorMovieDetails.invoke("Provide the details of the movie Avengers")



MovieDetails(title='Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye')], genres=['Action', 'Science Fiction'], budget=None)

### Typed Dict

It provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

The Validation is not compulsary here but it was in basemodel 
like in Integer if i give string output its fine

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str,"The title of the movie"]
    year:  Annotated[str,"The year the movie was released"]
    director: Annotated[str,"Director of the movie"]
    rating: Annotated[str,"Rating out of 10"]

model_with_typeDict = model.with_structured_output(MovieDict)
model_with_typeDict.invoke("Provide details about the movie Avengers")

{'director': 'Joss Whedon',
 'rating': '8.0',
 'title': 'Avengers',
 'year': '2012'}

In [11]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year : int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, "Budget in million USD"]

ActorMovieDetails = model.with_structured_output(MovieDetails)
ActorMovieDetails.invoke("Provide the details of the movie Avengers")



{'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [15]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Data Classes

A data class is a typically containing mainly data, although there aren't really any restrictions. You can create it using @dataclass decorator

In [17]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [19]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [21]:

from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: Annotated[str,"The name of the person"]
    email:Annotated[str,"The email address of the person"]
    phone:Annotated[str,"The phone number of the person"]


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [23]:
## Data Class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str # The name of the person"
    email: str # The email address of the person"
    phone: str # The phone number of the person"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
